# Анализ производительности реализаций кеша
Сравнение `IMemoryCache`, `IDistributedCache` (Redis, Valkey, Garnet) и `HybridCache`.

In [9]:
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import re

CSV_PATH = "../RequestMonitoring.Benchmarks/bin/Release/net10.0/BenchmarkDotNet.Artifacts/results/RequestMonitoring.Benchmarks.CacheBenchmark-report.csv"

df = pd.read_csv(CSV_PATH, sep=";")
df = df[["Method", "Mean", "Error", "StdDev", "Allocated"]].copy()

def parse_ns(val):
    # Убираем кавычки, запятые, единицы измерения (ns, μs, ms и т.д.)
    val = str(val).replace('"', '').replace(',', '').strip()
    val = re.sub(r'[a-zA-Z\s]', '', val)
    return float(val)

for col in ["Mean", "Error", "StdDev"]:
    df[col] = df[col].apply(parse_ns)

df["Allocated_B"] = df["Allocated"].astype(str).str.replace(' B', '').str.replace(',', '').str.strip().astype(float)
df["Operation"] = df["Method"].apply(lambda x: "Get" if "Get" in x else "Set")
df["Cache"] = df["Method"].str.strip("'")
df["Mean_us"] = df["Mean"] / 1000
df["Error_us"] = df["Error"] / 1000

df.head(10)

,Method,Mean,Error,StdDev,Allocated,Allocated_B,Operation,Cache,Mean_us,Error_us
0,'IMemoryCache Get',29.00,0.282,0.263,0 B,0.0,Get,IMemoryCache Get,0.02900,0.000282
1,'HybridCache Get',63.47,0.400,0.312,72 B,72.0,Get,HybridCache Get,0.06347,0.000400
2,'IMemoryCache Set',65.21,1.335,2.118,104 B,104.0,Set,IMemoryCache Set,0.06521,0.001335
3,'IDistributedCache Valkey Get',367561.48,6284.782,5878.789,1264 B,1264.0,Get,IDistributedCache Valkey Get,367.56148,6.284782
4,'IDistributedCache Redis Set',370856.88,7322.556,13931.925,1496 B,1496.0,Set,IDistributedCache Redis Set,370.85688,7.322556
5,'IDistributedCache Valkey Set',370878.72,7252.822,10401.781,1496 B,1496.0,Set,IDistributedCache Valkey Set,370.87872,7.252822
6,'IDistributedCache Redis Get',371111.16,6391.124,11846.359,1264 B,1264.0,Get,IDistributedCache Redis Get,371.11116,6.391124
7,'HybridCache Set',374357.71,7012.352,6216.267,2264 B,2264.0,Set,HybridCache Set,374.35771,7.012352
8,'IDistributedCache Garnet Get',414227.07,7567.870,10358.986,1264 B,1264.0,Get,IDistributedCache Garnet Get,414.22707,7.567870
9,'IDistributedCache Garnet Set',416453.13,8270.683,6906.395,1496 B,1496.0,Set,IDistributedCache Garnet Set,416.45313,8.270683


In [10]:
# График 1: Mean latency Get vs Set (логарифмическая шкала)

fig = make_subplots(rows=1, cols=2, subplot_titles=("Get", "Set"))
colors = px.colors.qualitative.Plotly

for i, op in enumerate(["Get", "Set"]):
    subset = df[df["Operation"] == op].sort_values("Mean")
    fig.add_trace(
        go.Bar(
            x=subset["Cache"],
            y=subset["Mean_us"],
            error_y=dict(type="data", array=subset["Error_us"].tolist()),
            name=op,
            marker_color=colors[:len(subset)],
            showlegend=False
        ),
        row=1, col=i+1
    )

fig.update_yaxes(title_text="Среднее время (μs)", type="log")
fig.update_xaxes(tickangle=-30)
fig.update_layout(title_text="Latency реализаций кеша (логарифмическая шкала)", height=500)
fig.show()

In [11]:
# График 2: Сравнение только distributed кешей (Redis, Valkey, Garnet)

distributed = df[df["Cache"].str.contains("Redis|Valkey|Garnet")].copy()

fig2 = px.bar(
    distributed.sort_values("Mean"),
    x="Cache",
    y="Mean_us",
    error_y="Error_us",
    color="Operation",
    barmode="group",
    title="Сравнение Redis / Valkey / Garnet",
    labels={"Mean_us": "Среднее время (μs)", "Cache": ""}
)
fig2.update_xaxes(tickangle=-30)
fig2.show()

In [12]:
# График 3: Аллокации памяти

fig3 = px.bar(
    df.sort_values("Allocated_B"),
    x="Cache",
    y="Allocated_B",
    color="Operation",
    barmode="group",
    title="Аллокации памяти на операцию",
    labels={"Allocated_B": "Байт", "Cache": ""}
)
fig3.update_xaxes(tickangle=-30)
fig3.show()

In [13]:
# Итоговая таблица

summary = df[["Cache", "Operation", "Mean_us", "Error_us", "Allocated_B"]].copy()
summary.columns = ["Реализация", "Операция", "Среднее (μs)", "Погрешность (μs)", "Аллокации (B)"]
summary = summary.sort_values("Среднее (μs)")
summary.style.background_gradient(subset=["Среднее (μs)"], cmap="RdYlGn_r")

,Реализация,Операция,Среднее (μs),Погрешность (μs),Аллокации (B)
0,IMemoryCache Get,Get,0.029000,0.000282,0.000000
1,HybridCache Get,Get,0.063470,0.000400,72.000000
2,IMemoryCache Set,Set,0.065210,0.001335,104.000000
3,IDistributedCache Valkey Get,Get,367.561480,6.284782,1264.000000
4,IDistributedCache Redis Set,Set,370.856880,7.322556,1496.000000
5,IDistributedCache Valkey Set,Set,370.878720,7.252822,1496.000000
6,IDistributedCache Redis Get,Get,371.111160,6.391124,1264.000000
7,HybridCache Set,Set,374.357710,7.012352,2264.000000
8,IDistributedCache Garnet Get,Get,414.227070,7.567870,1264.000000
9,IDistributedCache Garnet Set,Set,416.453130,8.270683,1496.000000
